In [0]:

raw_df = spark.sql('select * from dataco.bronze.raw_supply_chain')
display(raw_df)

In [0]:
display(raw_df.describe())

In [0]:
# Save DataFrame directly as a Managed Table (No path required)
# silver_source_df.write \
#     .format("delta") \
#     .mode("overwrite") \
#     .option("checkpointLocation", "/Volumes/dataco/landing/silver/supply_chain/_checkpoints")\
#     .saveAsTable("dataco.silver.enriched_supply_chain")

In [0]:
from pyspark.sql.functions import col, to_timestamp,regexp_replace,when
from delta.tables import DeltaTable

# 1. Read directly from the raw Delta table
bronze_df = spark.read.table("dataco.bronze.raw_supply_chain")

# Automatically replace spaces and remove parentheses from ALL column names
cleaned_cols_df = bronze_df
for col_name in bronze_df.columns:
    # Example transform: "order date (DateOrders)" or "order_date_(DateOrders)" -> "order_date_DateOrders"
    new_col_name = (
        col_name.replace(" ", "_")
                .replace("(", "")
                .replace(")", "")
                .replace("__", "_")  # Clean up any double underscores created in the process
    )
    cleaned_cols_df = cleaned_cols_df.withColumnRenamed(col_name, new_col_name)

# 3. Parse Dates and Cast Numerical / Price Fields
silver_source_df = (
    cleaned_cols_df
    # --- DATE / TIMESTAMP PARSING ---
    .withColumn("order_date_DateOrders", to_timestamp(col("order_date_DateOrders"), "M/d/yyyy H:mm"))
    .withColumn("shipping_date_DateOrders", to_timestamp(col("shipping_date_DateOrders"), "M/d/yyyy H:mm"))
    
    # --- PRICE / FINANCIAL FIELDS (CAST TO FLOAT) ---
    .withColumn("Benefit_per_order", col("Benefit_per_order").cast("float"))
    .withColumn("Sales_per_customer", col("Sales_per_customer").cast("float"))
    .withColumn("Order_Item_Discount", col("Order_Item_Discount").cast("float"))
    .withColumn("Order_Item_Discount_Rate", col("Order_Item_Discount_Rate").cast("float"))
    .withColumn("Order_Item_Product_Price", col("Order_Item_Product_Price").cast("float"))
    .withColumn("Order_Item_Profit_Ratio", col("Order_Item_Profit_Ratio").cast("float"))
    .withColumn("Sales", col("Sales").cast("float"))
    .withColumn("Order_Item_Total", col("Order_Item_Total").cast("float"))
    .withColumn("Order_Profit_Per_Order", col("Order_Profit_Per_Order").cast("float"))
    .withColumn("Product_Price", col("Product_Price").cast("float"))
    
    # --- OTHER NUMERICAL FIELDS (CAST TO INT) ---
    .withColumn("Days_for_shipping_real", col("Days_for_shipping_real").cast("integer"))
    .withColumn("Days_for_shipment_scheduled", col("Days_for_shipment_scheduled").cast("integer"))
    .withColumn("Category_Id", col("Category_Id").cast("integer"))
    .withColumn("Customer_Id", col("Customer_Id").cast("integer"))
    .withColumn("Department_Id", col("Department_Id").cast("integer"))
    .withColumn("Order_Customer_Id", col("Order_Customer_Id").cast("integer"))
    .withColumn("Order_Id", col("Order_Id").cast("integer"))
    .withColumn("Order_Item_Cardprod_Id", col("Order_Item_Cardprod_Id").cast("integer"))
    .withColumn("Order_Item_Id", col("Order_Item_Id").cast("integer"))
    .withColumn("Order_Item_Quantity", col("Order_Item_Quantity").cast("integer"))
    .withColumn("Product_Card_Id", col("Product_Card_Id").cast("integer"))
    .withColumn("Product_Category_Id", col("Product_Category_Id").cast("integer"))
    .withColumn(
        "Order_City",
        regexp_replace("Order_City", "\ufffd", "")
    )\
    .withColumn(
        "Order_State",
        regexp_replace("Order_State", "\ufffd", "")
    )
    .withColumn(
    "Order_Country",
    when(
        col("Order_Country").startswith("Espa"),
        "Espana"
    )
    .otherwise(
        regexp_replace(
            col("Order_Country"),
            "\ufffd",
            ""
        )
    )
)
)

# 4. Upsert (Merge) into Silver Delta Table
table_name = "dataco.silver.enriched_supply_chain"
silver_path = "/Volumes/dataco/landing/silver/checkpoint/enriched_supply_chain"

if spark.catalog.tableExists(table_name):
    silver_table = DeltaTable.forName(spark, table_name)
    (
        silver_table.alias("target")
        .merge(
            silver_source_df.alias("source"),
            "target.Order_Item_Id = source.Order_Item_Id"
        )
        .whenMatchedUpdateAll()     # Updates matching records
        .whenNotMatchedInsertAll()  # Inserts new records
        .execute()
    )
    print(f"Merged transformed data into {table_name}")
else:
    (
        silver_source_df.write
        .format("delta")
        .mode("overwrite")
        .option("path", silver_path)
        .saveAsTable(table_name)
    )
    print(f"Created new Silver table {table_name}")

In [0]:
%sql
SELECT * FROM dataco.silver.enriched_supply_chain